# 01 — Data Preparation
Load BTC/USDT 15m OHLCV from local CSV (Binance klines format),
resample to 1H and 4H, clean, and save to parquet on Google Drive.

In [ ]:
# Install dependencies
# torch, numpy, pandas are pre-installed by Colab — do NOT reinstall them.
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tensorboard tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ─── Project lives on Google Drive — no copying needed ─────
# Upload your project folder to:  My Drive / scalp2 / repo /
REPO_DIR = '/content/drive/MyDrive/scalp2/repo'

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2')):
    raise FileNotFoundError(
        'Project not found!\n'
        'Upload your project folder to Google Drive at:\n'
        '  My Drive → scalp2 → repo\n'
        'The "repo" folder should contain: scalp2/, config.yaml, data/'
    )

sys.path.insert(0, REPO_DIR)
print(f'Using project at {REPO_DIR}')

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')

# Output dirs on Google Drive for persistence
config.data.processed_dir = '/content/drive/MyDrive/scalp2/data/processed'
os.makedirs(config.data.processed_dir, exist_ok=True)

print(f'Symbol: {config.data.symbol}')
print(f'Date range: {config.data.date_range.start} to {config.data.date_range.end}')
print(f'Timeframes: {config.data.timeframes.primary}, {config.data.timeframes.mtf}')

In [ ]:
from scalp2.data.preprocessing import load_binance_csv, resample_ohlcv

# Load the 15m CSV (included in the repo under data/)
CSV_PATH = f'{REPO_DIR}/data/btcusdt_15min.csv'
df_15m = load_binance_csv(CSV_PATH)

print(f'15m: {len(df_15m):,} bars')
print(f'Columns: {list(df_15m.columns)}')
print(f'Range: {df_15m.index[0]} → {df_15m.index[-1]}')
df_15m.head()

In [ ]:
# Resample to 1H and 4H from 15m
df_1h = resample_ohlcv(df_15m, '1h')
df_4h = resample_ohlcv(df_15m, '4h')

print(f'1h: {len(df_1h):,} bars')
print(f'4h: {len(df_4h):,} bars')

In [ ]:
# Clean all timeframes (gap-fill, validate, optimize dtypes)
from scalp2.data.preprocessing import clean_ohlcv

data = {'15m': df_15m, '1h': df_1h, '4h': df_4h}

for tf in data:
    data[tf] = clean_ohlcv(data[tf], tf)
    print(f'{tf} after cleaning: {len(data[tf]):,} bars')

In [ ]:
# Save cleaned data to Google Drive
for tf, df in data.items():
    path = f'{config.data.processed_dir}/BTC_USDT_{tf}_clean.parquet'
    df.to_parquet(path)
    print(f'Saved {path}  ({len(df):,} bars)')

print('\nData preparation complete.')